# weedenroll — paper run on Google Colab (A100)

Runs the complete real-data pipeline of *Train Once, Enroll in the Field: Offline Few-Shot Weed Recognition with Prototype Enrollment on UAV-Class Edge Hardware*:
download the three public datasets → train the backbone (ResNet-50 teacher → distilled MobileNetV3-Small student) → full benchmark → figures, tables and `paper_assets.docx`.

**Runtime → Change runtime type → GPU (A100).**  Total time on an A100 is about 1 to 1.5 h, most of it the 1.7 GB Plant Seedlings download and the 45 training epochs.
Everything is saved to Google Drive as it goes so a disconnected session loses nothing that was already finished.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/weedenroll_paper'   # all results end up here
os.makedirs(DRIVE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code
Upload `weedenroll-src.zip` (the project folder zipped, from your machine) when prompted, **or** put it in the Drive folder above first and this cell picks it up.

In [ ]:
import os, zipfile, shutil
zip_path = os.path.join(DRIVE, 'weedenroll-src.zip')
if not os.path.exists(zip_path):
    from google.colab import files
    up = files.upload()                      # choose weedenroll-src.zip
    zip_path = next(iter(up))
shutil.rmtree('/content/weedenroll', ignore_errors=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content')
%cd /content/weedenroll
!ls

In [ ]:
# Colab already ships a CUDA build of torch/torchvision; install the rest.
!pip install -q -e ".[dev,docx,onnx,data]"
!python -m pytest -q
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 2. Data (no accounts needed)
DeepWeeds and Plant Seedlings V2 come from the Hugging Face mirrors, CoFly-WeedDB from Zenodo. The step ends with a count check against the published numbers and writes `data/verify.json` (Table 1).

In [ ]:
# reuse data already on Drive from a previous session, otherwise download (~3.4 GB)
if os.path.isdir(os.path.join(DRIVE, 'data')) and not os.path.isdir('data'):
    !cp -r "$DRIVE/data" data
!weedenroll data --dst data --all
!cat data/verify.json | head -40
if not os.path.isdir(os.path.join(DRIVE, 'data')):
    !rsync -a --exclude raw data "$DRIVE/"      # keep the converted ImageFolders (not the raw archives) on Drive

## 3. Train the backbone once
Base species of DeepWeeds only; `Prickly acacia`, `Siam weed`, `Snake weed` are held out (never trained on). Plant Seedlings and CoFly are never used for training.

In [ ]:
!weedenroll train --config configs/train.yaml
!ls -la weights
!cp -r weights "$DRIVE/" 

## 4. The paper run
Benchmarks (DR1, DR4–DR7), cost model (DR8), six figures, five tables, DOCX bundle. DR3 (footprint) is measured here. DR2 (latency, power) is reported as *projected*: the host-CPU latency is an upper bound and the Orin Nano row of Table 2 comes from published benchmarks. It becomes *measured* only if `weedenroll profile` is run on a device and `outputs/hardware_results.json` is added (step 6).

In [ ]:
!weedenroll run --config configs/real.yaml
!ls outputs/real outputs/real/figures outputs/real/tables

In [ ]:
!cat outputs/real/tables/table5_scorecard.md
!cat outputs/real/tables/table3_results.md

## 5. Save everything

In [ ]:
!cp -r outputs/real "$DRIVE/outputs_real"
!cd outputs && zip -qr /content/weedenroll_paper_outputs.zip real
!cp /content/weedenroll_paper_outputs.zip "$DRIVE/"
from google.colab import files
files.download('/content/weedenroll_paper_outputs.zip')
print('saved to', DRIVE)

## 6. Optional: measured device numbers
On the Jetson Orin Nano copy `weights/student.onnx`, run `weedenroll profile --config configs/jetson.yaml --out outputs/hardware_results.json`, put that JSON in `outputs/` here (or on Drive) and re-run step 4 (or `weedenroll report --config configs/real.yaml`); Table 2 and the DR2 row switch from projected to measured.